In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from tqdm import tqdm
import re
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, precision_score, recall_score 
import lightgbm as lgb
import pickle
from sklearn.preprocessing import LabelEncoder
import datetime

In [2]:
CSV_PATH = "../csv/"
K_COLS = ['着', 'RaceID', '選手登番', '展示']
DATE = "221122"

In [3]:
k_files = glob.glob("../csv/K" + DATE + ".csv")
b_files = glob.glob("../csv/B" + DATE + ".csv")
def concat_files(files):
    tmp = pd.DataFrame()
    for file in tqdm(files):
        df = pd.read_csv(file, index_col=0)
        tmp = pd.concat([tmp,df])
    return tmp
kdf = concat_files(k_files)
bdf = concat_files(b_files)

100%|██████████| 1/1 [00:00<00:00, 84.51it/s]


In [4]:
df = pd.merge(bdf,kdf.loc[:,K_COLS],on = ['RaceID','選手登番'])

In [5]:
def LabelEncoding(df,col):
    #インスタンス
    LE = LabelEncoder()

    #Label エンコーディング
    LE.fit_transform(df[col].values)

    pickle.dump(LE, open('../model/' + col + '_LE.pickle', 'wb'))

    #データ変換
    le = LE.fit_transform(df[col].values)
    return le

def zero_padding(txt):
    l = re.findall(r"\d+", txt)
    l = [int(s) for s in l]
    date = datetime.date(*l[:3])
    raceid = str(date) + '-' + str(l[3]).zfill(2) + '-' + str(l[4]).zfill(2)
    return raceid

In [6]:
df_encoded = df
df_encoded['支部'] = LabelEncoding(df,'支部')
df_encoded['級別'] = LabelEncoding(df, '級別')
df_encoded['R'] = df['R'].replace('R', '', regex=True).astype('int')
df_encoded['RaceID'] = df_encoded['RaceID'].replace('_','/',regex=True).replace('R','',regex=True)

In [7]:
zero = []
for n,i in enumerate(tqdm(df_encoded['RaceID'])):
    zero.append(zero_padding(i))
df_encoded['RaceID'] = zero

100%|██████████| 782/782 [00:00<00:00, 90269.59it/s]


In [8]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 782 entries, 0 to 781
Data columns (total 18 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   艇番       782 non-null    int64  
 1   選手登番     782 non-null    int64  
 2   選手名      782 non-null    object 
 3   年齢       782 non-null    int64  
 4   支部       782 non-null    int64  
 5   体重       782 non-null    int64  
 6   級別       782 non-null    int64  
 7   全国勝率     782 non-null    float64
 8   全国2連率    782 non-null    float64
 9   当地勝率     782 non-null    float64
 10  当地2連率    782 non-null    float64
 11  モーター2連率  782 non-null    float64
 12  ボート2連率   782 non-null    float64
 13  RaceID   782 non-null    object 
 14  場所       782 non-null    int64  
 15  R        782 non-null    int64  
 16  着        782 non-null    int64  
 17  展示       782 non-null    float64
dtypes: float64(7), int64(9), object(2)
memory usage: 116.1+ KB


In [9]:
df_test = df_encoded.drop(['選手名','着','RaceID'],axis=1)

In [12]:
with open('../model/lgb_clf.pickle', 'rb') as f:
    lgb_clf = pickle.load(f)
df_pred = lgb_clf.predict(df_test ,num_iteration=lgb_clf.best_iteration)

In [13]:
df_encoded["pred"] = df_pred


In [14]:
df_encoded

,艇番,選手登番,選手名,年齢,支部,体重,級別,全国勝率,全国2連率,当地勝率,当地2連率,モーター2連率,ボート2連率,RaceID,場所,R,着,展示,pred
0,1,4515,藤田浩人,36,1,54,1,5.64,36.57,6.07,42.86,37.50,31.76,2022-11-22-22-01,22,1,3,6.81,1.496566
1,2,4222,山口高志,40,1,52,2,4.24,27.50,5.49,41.27,30.86,22.78,2022-11-22-22-01,22,1,5,6.78,-0.481119
2,3,3662,大庭元明,52,13,56,0,5.33,38.13,6.64,52.78,32.58,26.83,2022-11-22-22-01,22,1,1,6.81,-0.045768
3,4,3516,吉原聖人,55,7,53,2,4.36,22.83,3.76,16.00,20.00,44.87,2022-11-22-22-01,22,1,2,6.85,-1.391673
4,5,3614,谷勝幸,49,7,54,2,4.26,22.58,4.78,29.27,26.85,30.00,2022-11-22-22-01,22,1,4,6.87,-1.892718
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
777,2,4174,赤坂俊輔,40,15,52,0,7.08,57.84,7.25,62.50,34.30,0.00,2022-11-22-01-12,1,12,3,6.76,0.658650
778,3,3983,須藤博倫,45,3,52,0,6.56,46.55,6.16,41.82,34.81,0.00,2022-11-22-01-12,1,12,6,6.72,0.432288
779,4,4645,上村純一,42,14,54,0,6.32,49.57,6.37,46.34,37.05,0.00,2022-11-22-01-12,1,12,1,6.75,0.303586
780,5,4276,鈴木勝博,39,9,52,0,7.29,65.52,7.46,64.41,32.51,0.00,2022-11-22-01-12,1,12,4,6.72,0.377937


In [15]:
pred_rank = pd.DataFrame()
for i in df_encoded["RaceID"].unique():
    prd = df_encoded[df_encoded["RaceID"] == i]["pred"].rank(ascending=False)
    pred_rank = pd.concat([pred_rank,prd])

In [16]:
df_encoded["pred_rank"] = pred_rank

In [17]:
places = df_encoded['場所'].unique()
place_list = {
            '桐生':'01',
            '戸田':'02',
            '江戸川':'03',
            '平和島':'04',
            '多摩川':'05',
            '浜名湖':'06',
            '蒲郡':'07',
            '常滑':'08',
            '津':'09',
            '三国':'10',
            'びわこ':'11',
            '住之江':'12',
            '尼崎':'13',
            '鳴門':'14',
            '丸亀':'15',
            '児島':'16',
            '宮島':'17',
            '徳山':'18',
            '下関':'19',
            '若松':'20',
            '芦屋':'21',
            '福岡':'22',
            '唐津':'23',
            '大村':'24'}
def get_keys_from_value(d, val):
    return [k for k, v in d.items() if v == val]
for p in places:
    if p < 10:
        v = '0' + str(p)
        PLACE = get_keys_from_value(place_list,v)[0]
    else:
        v = str(p)
        PLACE = get_keys_from_value(place_list,v)[0]
    os.makedirs('../prediction/' + DATE + '/',exist_ok=True)
    with open("../prediction/"+DATE+"/"+ DATE+"_"+ PLACE +".txt", "w"):
        pass
    f = open("../prediction/"+DATE+"/"+ DATE+"_"+ PLACE +".txt", "w")
    print(PLACE,file=f)
    print(PLACE)
    y_predp = df_encoded[df_encoded['場所'] == p]
    r_list = y_predp['R'].unique()
    
    print('-----------------------',file=f)
    for i in r_list:
        print("第"+str(i) + "R",file=f)
        print("第"+str(i) + "R")
        race_pd = y_predp[y_predp["R"] == i]
        first = race_pd[race_pd['pred_rank'] == 1]['艇番'].iloc[0]
        second = race_pd[race_pd['pred_rank'] == 2]['艇番'].iloc[0]
        third = race_pd[race_pd['pred_rank'] == 3]['艇番'].iloc[0]
        print(str(first)+'-'+str(second) +'-' + str(third),file=f)
        print(str(first)+'-'+str(second) +'-' + str(third))
    print('-----------------------\n',file=f)
    print('-----------------------\n')
    f.close()

福岡
第1R
1-3-2
第2R
1-2-5
第3R
1-3-2
第4R
5-3-1
第5R
1-3-5
第6R
4-3-1
第7R
1-4-2
第8R
1-2-4
第9R
1-3-2
第10R
2-1-5
第11R
1-3-2
第12R
1-3-2
-----------------------

芦屋
第1R
1-3-4
第2R
1-4-3
第3R
1-3-2
第4R
1-4-3
第5R
1-4-2
第6R
2-1-4
第7R
1-4-5
第8R
1-2-3
第9R
1-2-3
第10R
1-3-2
第11R
1-5-3
第12R
1-4-2
-----------------------

下関
第1R
2-1-5
第2R
1-2-4
第3R
1-4-3
第4R
2-3-1
第5R
1-3-2
第6R
1-3-6
第7R
1-2-4
第8R
1-4-2
第9R
1-4-2
第10R
1-2-3
第11R
1-2-5
第12R
1-2-3
-----------------------

丸亀
第1R
1-2-4
第2R
3-5-1
第3R
1-3-2
第4R
5-3-1
第5R
1-2-5
第6R
1-3-2
第7R
1-4-6
第8R
1-4-2
第9R
1-2-6
第10R
1-3-2
第11R
1-3-2
第12R
1-4-5
-----------------------

鳴門
第1R
1-4-3
第2R
1-4-3
第3R
1-2-3
第4R
1-2-3
第5R
1-3-2
第6R
1-2-4
第7R
1-3-2
第8R
1-2-4
第9R
1-3-2
第10R
1-2-3
第11R
1-3-2
第12R
1-4-2
-----------------------

住之江
第1R
1-2-3
第2R
2-1-4
第3R
4-3-2
第4R
1-5-4
第5R
1-3-6
第6R
1-3-5
第7R
1-4-3
第8R
1-3-5
第9R
1-2-3
第10R
1-2-4
第11R
1-2-3
第12R
1-2-3
-----------------------

三国
第1R
1-3-4
第2R
2-1-3
第3R
1-3-2
第4R
1-5-3
第5R
1-4-2
第6R
3-2-1
第7R
1-2-3
第8R
1-3-2
第9R
1-2-4
